<div dir="rtl">

# NB06-M01 — Tafsir Inventory

## هدف
ساخت Inventory قابل دفاع برای المیزان، تسنیم، نمونه، نورالثقلین و البرهان پیش از هر نوع Crawl یا Ingestion.

این Notebook باید دقیقاً نشان دهد چه منبعی پیدا شده، چه چیزی هنوز پیدا نشده، چه نسخه‌ای هدف است و کدام مانع اجازه ورود متن به Corpus را نمی‌دهد.

</div>

In [1]:
# NB06-C01 — Imports and root
import json
import sys
from pathlib import Path

current = Path.cwd().resolve()
project_root = next(
    (p for p in [current, *current.parents] if (p / ".git").exists() and (p / "data").exists()),
    None,
)
if project_root is None:
    raise RuntimeError("QRAG project root not found.")

src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from qrag_phase1.foundation import collect_git_info, write_json_atomic

assert collect_git_info(project_root)["branch"] == "qrag-phase1"

<div dir="rtl">

# NB06-M02 — Load Inventory Contract

Inventory شامل Sourceهای پیدا‌شده و Blockerهای صریح است. URL پیدا‌شده به معنی مجوز Crawl نیست.

</div>

In [2]:
# NB06-C02 — Load tafsir inventory and acquisition plan
inventory_path = project_root / "data/phase1/manifests/tafsir_inventory_v1.json"
plan_path = project_root / "config/phase1/tafsir_acquisition_plan_v1.json"
schema_path = project_root / "schemas/phase1/tafsir_source.schema.json"

for path in (inventory_path, plan_path, schema_path):
    assert path.is_file(), f"Missing input: {path}"

inventory = json.loads(inventory_path.read_text(encoding="utf-8"))
plan = json.loads(plan_path.read_text(encoding="utf-8"))
schema = json.loads(schema_path.read_text(encoding="utf-8"))

assert len(inventory["sources"]) == 5
assert plan["blockers"]

<div dir="rtl">

# NB06-M03 — Source Identity Checks

هیچ Source تکراری یا بدون Author/Method/Locator policy پذیرفته نمی‌شود.

</div>

In [3]:
# NB06-C03 — Identity and completeness checks
tafsir_ids = [source["tafsir_id"] for source in inventory["sources"]]
assert len(tafsir_ids) == len(set(tafsir_ids))

for source in inventory["sources"]:
    assert source["title"].strip()
    assert source["author"].strip()
    assert source["tafsir_method"].strip()
    assert source["locator_requirements"]
    assert source["acceptance_status"] == "candidate_external"
    assert source["rights_status"] == "evidence_required"

print("Tafsir identity checks: PASS")

Tafsir identity checks: PASS


<div dir="rtl">

# NB06-M04 — Priority and Riwayi Separation

المیزان/تسنیم/نمونه به‌عنوان لایه تحلیلی و نورالثقلین/البرهان به‌عنوان Tafsir روایی جدا ثبت می‌شوند.

</div>

In [4]:
# NB06-C04 — Method-family separation
analytical = [
    source["tafsir_id"]
    for source in inventory["sources"]
    if "riwayi" not in source["tafsir_method"]
]
riwayi = [
    source["tafsir_id"]
    for source in inventory["sources"]
    if "riwayi" in source["tafsir_method"]
]

assert "tafsir:al-mizan" in analytical
assert "tafsir:tasnim" in analytical
assert "tafsir:nemuneh" in analytical
assert "tafsir:nur-al-thaqalayn" in riwayi
assert "tafsir:al-burhan" in riwayi

print("Analytical:", analytical)
print("Riwayi:", riwayi)

Analytical: ['tafsir:al-mizan', 'tafsir:tasnim', 'tafsir:nemuneh']
Riwayi: ['tafsir:nur-al-thaqalayn', 'tafsir:al-burhan']


<div dir="rtl">

# NB06-M05 — Acquisition Readiness

فقط Sourceهایی که Edition، rights و locator آن‌ها روشن است Ready محسوب می‌شوند. در Wave 04 انتظار Ready=0 داریم.

</div>

In [5]:
# NB06-C05 — Acquisition readiness
ready = []
blocked = []

for source in inventory["sources"]:
    has_candidate = bool(source["candidate_sources"])
    rights_verified = source["rights_status"] == "verified_allowed"
    acquired = source["acquisition_status"] == "verified_snapshot"

    if has_candidate and rights_verified and acquired:
        ready.append(source["tafsir_id"])
    else:
        blocked.append({
            "tafsir_id": source["tafsir_id"],
            "has_candidate_source": has_candidate,
            "rights_verified": rights_verified,
            "verified_snapshot": acquired,
            "acquisition_status": source["acquisition_status"],
        })

assert ready == []
print(json.dumps(blocked, ensure_ascii=False, indent=2))

[
  {
    "tafsir_id": "tafsir:al-mizan",
    "has_candidate_source": true,
    "rights_verified": false,
    "verified_snapshot": false,
    "acquisition_status": "target_persian_or_arabic_edition_not_acquired"
  },
  {
    "tafsir_id": "tafsir:tasnim",
    "has_candidate_source": false,
    "rights_verified": false,
    "verified_snapshot": false,
    "acquisition_status": "authoritative_machine_readable_source_not_verified"
  },
  {
    "tafsir_id": "tafsir:nemuneh",
    "has_candidate_source": false,
    "rights_verified": false,
    "verified_snapshot": false,
    "acquisition_status": "authoritative_machine_readable_source_not_verified"
  },
  {
    "tafsir_id": "tafsir:nur-al-thaqalayn",
    "has_candidate_source": true,
    "rights_verified": false,
    "verified_snapshot": false,
    "acquisition_status": "candidate_sources_discovered_not_acquired"
  },
  {
    "tafsir_id": "tafsir:al-burhan",
    "has_candidate_source": true,
    "rights_verified": false,
    "verified_snapsh

<div dir="rtl">

# NB06-M06 — Acceptance Report

هدف Inventory می‌تواند محقق شود حتی اگر Ingestion هنوز Blocked باشد.

</div>

In [6]:
# NB06-C06 — Persist inventory acceptance
reports = project_root / "reports/phase1"
reports.mkdir(parents=True, exist_ok=True)

acceptance = {
    "schema_version": "qrag_nb06_acceptance_v1",
    "objective_status": "ACHIEVED",
    "acceptance_status": "PASS_WITH_LIMITATIONS",
    "failed_checks": [],
    "candidate_tafsir_count": len(inventory["sources"]),
    "ingestion_ready_count": len(ready),
    "blocked_sources": blocked,
    "global_blockers": plan["blockers"],
    "next_notebook": "notebooks/phase1/07_tafsir_ingestion_and_acceptance.ipynb",
    "next_notebook_execution_mode": "FRAMEWORK_AND_PREFLIGHT_ONLY_UNTIL_SOURCES_ACQUIRED",
}

path = reports / "tafsir_inventory_acceptance_v1.json"
write_json_atomic(path, acceptance)
assert json.loads(path.read_text(encoding="utf-8")) == acceptance
print(json.dumps(acceptance, ensure_ascii=False, indent=2))

{
  "schema_version": "qrag_nb06_acceptance_v1",
  "objective_status": "ACHIEVED",
  "acceptance_status": "PASS_WITH_LIMITATIONS",
  "failed_checks": [],
  "candidate_tafsir_count": 5,
  "ingestion_ready_count": 0,
  "blocked_sources": [
    {
      "tafsir_id": "tafsir:al-mizan",
      "has_candidate_source": true,
      "rights_verified": false,
      "verified_snapshot": false,
      "acquisition_status": "target_persian_or_arabic_edition_not_acquired"
    },
    {
      "tafsir_id": "tafsir:tasnim",
      "has_candidate_source": false,
      "rights_verified": false,
      "verified_snapshot": false,
      "acquisition_status": "authoritative_machine_readable_source_not_verified"
    },
    {
      "tafsir_id": "tafsir:nemuneh",
      "has_candidate_source": false,
      "rights_verified": false,
      "verified_snapshot": false,
      "acquisition_status": "authoritative_machine_readable_source_not_verified"
    },
    {
      "tafsir_id": "tafsir:nur-al-thaqalayn",
      "has_can

<div dir="rtl">

# NB06-M07 — جمع‌بندی

Wave 04 عمداً Crawl انجام نمی‌دهد. قبل از Ingestion باید Source واقعی، Edition، حقوق استفاده و Locator قابل بازسازی بسته شود.

</div>